In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib # For saving the scaler object

In [2]:
# 1. Load the data generated in Phases 2 and 3
# Train set represents "Normal" baseline (Phase 2)
train_X = np.load('baseline_normal.npy')

# Test set represents Normal + Anomalous data (Phase 3)
test_X = np.load('test_X.npy')
test_y = np.load('test_y.npy')

print(f"Loaded train_X shape: {train_X.shape}")
print(f"Loaded test_X shape:  {test_X.shape}")
print(f"Loaded test_y shape:  {test_y.shape}")

Loaded train_X shape: (72000, 9)
Loaded test_X shape:  (72000, 9)
Loaded test_y shape:  (72000,)


In [3]:
# 2. Initialize the Scaler
# MinMaxScaler scales everything to a [0, 1] range based on the training set
scaler = MinMaxScaler(feature_range=(0, 1))

# 3. Fit ONLY on training data, then transform both
print("Scaling features...")
train_X_scaled = scaler.fit_transform(train_X)
test_X_scaled = scaler.transform(test_X)

# Optional but recommended: Save the scaler! 
# In a real-world scenario, the live ESP32 data will need to be 
# transformed using these exact same min/max parameters.
joblib.dump(scaler, 'sensor_scaler.gz')
print("Saved scaler to sensor_scaler.gz")

# Sanity Check
print(f"Train Scaled Min: {train_X_scaled.min():.4f}, Max: {train_X_scaled.max():.4f}")
# Notice that test_X_scaled might have values > 1.0 or < 0.0. This is NORMAL and GOOD. 
# Anomalies (like temperature runaway) should blow past the 0-1 boundaries!
print(f"Test Scaled Min:  {test_X_scaled.min():.4f}, Max: {test_X_scaled.max():.4f}")

Scaling features...
Saved scaler to sensor_scaler.gz
Train Scaled Min: 0.0000, Max: 1.0000
Test Scaled Min:  -4.9175, Max: 6.7046


In [4]:
def create_sequences(data, labels=None, window_size=60, step=1):
    """
    Slices a 2D array into overlapping 3D sequences.
    If labels are provided, assigns a single label to the entire window.
    """
    sequences = []
    seq_labels = []
    
    # Iterate through the data with the specified step size
    for i in range(0, len(data) - window_size + 1, step):
        # Extract the window
        window = data[i : i + window_size]
        sequences.append(window)
        
        # Determine the label for this window
        if labels is not None:
            # If ANY point in this window is anomalous (>0), we flag the whole window as an anomaly.
            # Alternatively, you could just check the last point: labels[i + window_size - 1]
            window_label = np.max(labels[i : i + window_size])
            seq_labels.append(window_label)
            
    if labels is not None:
        return np.array(sequences, dtype=np.float32), np.array(seq_labels, dtype=np.int8)
    
    return np.array(sequences, dtype=np.float32)

# ── Configuration
WINDOW_SIZE = 60  # 3 seconds at 20 Hz
STEP_SIZE   = 10  # Slide forward by 0.5 seconds each time to save RAM

print(f"Creating sequences (Window={WINDOW_SIZE}, Step={STEP_SIZE})...")

# Create Training Sequences
X_train_seq = create_sequences(train_X_scaled, window_size=WINDOW_SIZE, step=STEP_SIZE)

# Create Testing Sequences & Windowed Labels
X_test_seq, y_test_seq = create_sequences(test_X_scaled, labels=test_y, window_size=WINDOW_SIZE, step=STEP_SIZE)

Creating sequences (Window=60, Step=10)...


In [5]:
# ── Verification
print("────────────────────────────────────────────────────────────")
print("  Phase 4 Complete: Tensors Ready for GRU")
print("────────────────────────────────────────────────────────────")
print(f"Original Train 2D: {train_X_scaled.shape}")
print(f"New Train 3D:      {X_train_seq.shape}")
print("")
print(f"Original Test 2D:  {test_X_scaled.shape}")
print(f"New Test 3D:       {X_test_seq.shape}")
print(f"New Test Labels:   {y_test_seq.shape}")
print("────────────────────────────────────────────────────────────")

# Understand the Shape: (7195, 60, 9)
# 7195 = Number of individual 3-second blocks
# 60   = Timesteps per block
# 9    = Sensor features per timestep

# ── Save the arrays
np.save('X_train_seq.npy', X_train_seq)
np.save('X_test_seq.npy', X_test_seq)
np.save('y_test_seq.npy', y_test_seq)

print("Saved → X_train_seq.npy")
print("Saved → X_test_seq.npy")
print("Saved → y_test_seq.npy")
print("\nNext step: Phase 5 — Building the GRU Autoencoder in TensorFlow/Keras or PyTorch.")

────────────────────────────────────────────────────────────
  Phase 4 Complete: Tensors Ready for GRU
────────────────────────────────────────────────────────────
Original Train 2D: (72000, 9)
New Train 3D:      (7195, 60, 9)

Original Test 2D:  (72000, 9)
New Test 3D:       (7195, 60, 9)
New Test Labels:   (7195,)
────────────────────────────────────────────────────────────
Saved → X_train_seq.npy
Saved → X_test_seq.npy
Saved → y_test_seq.npy

Next step: Phase 5 — Building the GRU Autoencoder in TensorFlow/Keras or PyTorch.
